# Agents

## Define a tool, build the agent with a "provider:model" string, and invoke it.

In [ ]:
import os
from dotenv import load_dotenv
from pydantic import SecretStr

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool

load_dotenv()

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

llm = ChatOpenAI(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

agent = create_agent(
    model=llm,
    tools=[search],
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

result = agent.invoke(
    {"messages": [{"role":"user", "content":"What is AI?"}]}
)

print(result.get("messages")[-1].content)

Artificial Intelligence (AI) is the field of computer science that designs systems capable of performing tasks that normally require human intelligence. These tasks include learning from data, reasoning, problem‑solving, perception (e.g., vision, speech), and natural‑language understanding. AI techniques range from symbolic methods (rules, logic) to statistical approaches such as machine‑learning, deep learning, and reinforcement learning. The ultimate goal is to create machines that can adapt, predict, and act autonomously in complex environments.


## Give the agent memory with a checkpointer, keyed by a thread id.

In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=llm,
    tools=[search],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "thread-123"}}
reply = agent.invoke({"messages": [{"role": "user", "content": "My name is Rajesh."}]}, config)
print(reply["messages"][-1].content)

Nice to meet you, Rajesh! How can I assist you today?


In [7]:
reply = agent.invoke({"messages": [{"role": "user", "content": "whats my name"}]}, config)
print(reply["messages"][-1].content)

Your name is Rajesh.


## Stream the loop step by step instead of waiting for the whole run.

In [10]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Search and summarize AI."}]},
    stream_mode="values", config=config):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

Search and summarize AI.
================================== Ai Message ==================================
Tool Calls:
  search (chatcmpl-tool-aa5b4a00c4f02415)
 Call ID: chatcmpl-tool-aa5b4a00c4f02415
  Args:
    query: Artificial Intelligence overview summary
================================= Tool Message =================================
Name: search

Results for: Artificial Intelligence overview summary
================================== Ai Message ==================================
Tool Calls:
  search (chatcmpl-tool-8fc38e84d4c36164)
 Call ID: chatcmpl-tool-8fc38e84d4c36164
  Args:
    query: Artificial intelligence definition, history, applications, challenges, future prospects
================================= Tool Message =================================
Name: search

Results for: Artificial intelligence definition, history, applications, challenges, future prospects
=============================